### Construction d'un dataframe,  d'un fichier csv et d'un json listant tous les pays avec pour chaque pays :
- codes iso 3166-1 alpha-2 et alpha-3
- noms anglais et français
- noms anglais et français plus affichables (plus concis pour les noms à rallonge)
- continent anglais et français
- pseudo code continent
- couleur attribuée à chaque continent
- drapeau en svg path

### Sources :
- extraction des codes alpha3 pays [owid-co2-data.csv] :
https://github.com/owid/co2-data/tree/master et https://nyc3.digitaloceanspaces.com/owid-public/data/co2/owid-co2-data.csv complété par les codes alpha2 des drapeaux de https://github.com/lipis/flag-icons/tree/main
- conversions code alpha /nom pays / continent : librairie python pycountry-convert
- noms des pays en français à partir des codes alpha2 [Nom_pays_FR_isocode.csv] :
- code svg des drapeaux [flags/...] :
ANCIEN CODE : https://github.com/hampusborgos/country-flags/, et pour télécharger :
https://github.com/hampusborgos/country-flags/archive/refs/heads/main.zip

NOUVEAU CODE : https://github.com/lipis/flag-icons/tree/main
### Sorties :
- 04-GRAPH_racing_bars/dataset_brut/world_countries.csv
- 03-GRAPH_geo_kml/dataset_source_country_name/country_name.json

## Fichiers sources (utilise les fichiers source de 04-GRAPH_racing_bars)

In [1]:
# nom des fichiers sources 
sourcePath = "dataset_source/"
sourceCSVFile = "owid-co2-data.csv"
sourceCSVFRnames = "Nom_pays_FR_isocode.csv" # correspondance iso_code - noms en français
sourceSVGFlags = "flag-icons_2023_update/"

## Chemin des fichiers produits 

In [2]:
# worldMapPath = "/home/DATA/ECOSKETCH/SiteWeb/APPS_DATAVIZ/SVGD3/03-GRAPH_geo_kml/dataset_source_country_name/"
# racingBarPath = "/home/DATA/ECOSKETCH/SiteWeb/APPS_DATAVIZ/SVGD3/04-GRAPH_racing_bars/dataset_brut/"
produitPath = "dataset_produit/"

## Importation des source et construction du dataframe

In [3]:
import pandas as pd
import pycountry_convert as pc
import os
import json

In [4]:

#=============================================================================================
# récupération des données des fichiers vers dataframes, listes ou dictionnaires
dataSource = pd.read_csv(sourcePath + sourceCSVFile)
alpha3_list = dataSource[~dataSource["iso_code"].isna()]["iso_code"].unique()
dataFRname = pd.read_csv(sourcePath + sourceCSVFRnames).set_index("ISO 3166-1 alpha-3")

# compléments de traduction :
countriesTradComplements = {
    'BES': 'Bonaire, Saint Eustache et Saba',
    'VGB': "Iles Vierges Britanniques",
    'CUW': 'Curaçao',
    'TLS': 'Timor Oriental',
    'ANT': 'Antilles néerlandaises'
}
for code in countriesTradComplements.keys():
    dataFRname.at[code, "Pays"] = countriesTradComplements[code] 
dataFRname[dataFRname.index.isin(countriesTradComplements.keys())]


flags = dict()
directory = sourcePath+sourceSVGFlags
for file in os.listdir(sourcePath+sourceSVGFlags):
    filename = os.path.join(directory, file)
    if filename[-4:] == ".svg":
        key = filename.split("/")[-1][:-4].upper()
        with open(directory+file, "r") as f:
            value = f.read()
        flags[key] = value
    else:
        print(" !!! nom pas pris en compte > ", filename.split("/")[-1])

 
#=============================================================================================
# terminologie française des continents
continents_trad = {
    'Africa': 'Afrique',
    'Asia': 'Asie', 
    'Europe': 'Europe', 
    'North America': 'Amérique du Nord', 
    'Oceania': 'Océanie',
    'South America': 'Amérique du Sud',
    'Antarctica': 'Antarctique',
    'no_name': 'no_name',
    'World': 'Monde'
}
# conversion continentCode -> ENcontinent, FRcontinent :
def continentCodeGeneration(continent):        
    l = continent.split(" ")
    return "".join([c[:4-2*len(l)+1+i].upper() for i, c in enumerate(l)])
    
continentId = { continentCodeGeneration(c):[c, continents_trad[c]] for c in continents_trad.keys()}

#=============================================================================================
# fonctions de correspondance entre les iso-codes et les noms de pays / continents
def alpha3_to_alpha2(alpha3):
    try:
        alpha2 = pc.country_alpha3_to_country_alpha2(alpha3)
        return alpha2
    except KeyError as Error:
        if alpha3 in alpha2Complements.keys():
            return alpha2Complements[alpha3]
        else:                
            return "no_alpha2"
        
def alpha3_to_ENname(alpha3):
    try:
        country_alpha2 = pc.country_alpha3_to_country_alpha2(alpha3)
        country_name = pc.country_alpha2_to_country_name(country_alpha2)
        return country_name
    except (KeyError, TypeError) as Error:
        if alpha3 in countriesComplements.keys():
            return countriesComplements[alpha3]
        else:                
            return "no_name"
        
def alpha3_to_FRname(alpha3):
    try:
        FRname = dataFRname.at[alpha3, "Pays"]
        return FRname
    except KeyError:
        return "no_name"
    
def alpha3_to_ENcontinent(alpha3):
    try:
        country_alpha2 = pc.country_alpha3_to_country_alpha2(alpha3)
        country_continent_code = pc.country_alpha2_to_continent_code(country_alpha2)
        country_continent_name = pc.convert_continent_code_to_continent_name(country_continent_code)
        return country_continent_name
    except (KeyError, TypeError) as Error:
        if alpha3 in continentsComplements.keys():
            return continentsComplements[alpha3]
        else:                
            return "no_name"

def alpha3_to_FRcontinent(alpha3):
    try:
        country_continent_name = alpha3_to_ENcontinent(alpha3)
        return continents_trad[country_continent_name]
    except (KeyError, TypeError) as Error:
        return "no_name"
    
def alpha3_to_continent_code(alpha3):
    try:
        continent = alpha3_to_ENcontinent(alpha3)
        if continent == "no_name":
            return "NON"
        return continentCodeGeneration(continent)
#         l = continent.split(" ")
#         return "".join([c[:4-2*len(l)+1+i].upper() for i, c in enumerate(l)])
    except (KeyError, TypeError) as Error:
        return "NON"

def alpha3_to_flag(alpha3):
    try:
        country_alpha2 = pc.country_alpha3_to_country_alpha2(alpha3)
        return flags[country_alpha2]
    except (KeyError, TypeError) as Error:
        if alpha3 in alpha2Complements.keys():
            try:
                return flags[alpha2Complements[alpha3]]
            except (KeyError, TypeError) as Error:
                return "no_flag"   
        else:                
            return "no_flag"   
        
#=============================================================================================
# spécial compléments : 
alpha2Complements = {"ANT": "AN"}
continentsComplements = {"ATA":"Antarctica", "TLS": "Asia", "SXM": "North America", "ESH": "Africa", "ANT": "North America"}
countriesComplements = {"ANT": "Netherlands Antilles"}
countriesTradComplements = {
    'BES':'Bonaire, Saint Eustache et Saba',
    'VGB': "Iles Vierges Britanniques",
    'CUW': 'Curaçao',
    'TLS': 'Timor Oriental',
    'ANT': 'Antilles néerlandaises'
}
flags
#=============================================================================================
# dataframe global

WorldCountries = pd.DataFrame(
    {
        "alpha3": alpha3_list,
        "alpha2": [alpha3_to_alpha2(alpha3) for alpha3 in alpha3_list],
        "ENname": [alpha3_to_ENname(alpha3) for alpha3 in alpha3_list],
        "FRname": [alpha3_to_FRname(alpha3) for alpha3 in alpha3_list],
        "ENcontinent": [alpha3_to_ENcontinent(alpha3) for alpha3 in alpha3_list],
        "FRcontinent": [alpha3_to_FRcontinent(alpha3) for alpha3 in alpha3_list],
        "continentCode": [alpha3_to_continent_code(alpha3) for alpha3 in alpha3_list],
        "svgFlag": [alpha3_to_flag(alpha3) for alpha3 in alpha3_list]
    },
).set_index("alpha3")
WorldCountries

,alpha2,ENname,FRname,ENcontinent,FRcontinent,continentCode,svgFlag
alpha3,,,,,,,
AFG,AF,Afghanistan,Afghanistan,Asia,Asie,ASI,"<svg xmlns=""http://www.w3.org/2000/svg"" xmlns:..."
ALA,AX,Åland Islands,Îles Åland,Europe,Europe,EUR,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl..."
ALB,AL,Albania,Albanie,Europe,Europe,EUR,"<svg xmlns=""http://www.w3.org/2000/svg"" xmlns:..."
DZA,DZ,Algeria,Algérie,Africa,Afrique,AFR,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl..."
ASM,AS,American Samoa,Samoa américaines,Oceania,Océanie,OCE,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl..."
...,...,...,...,...,...,...,...
WLF,WF,Wallis and Futuna,Wallis-et-Futuna,Oceania,Océanie,OCE,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl..."
ESH,EH,Western Sahara,Sahara occidental,Africa,Afrique,AFR,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl..."
YEM,YE,Yemen,Yémen,Asia,Asie,ASI,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl..."


##### Ajout de colonnes avec nom de pays plus "affichable" (plus court)

In [5]:


# afficher les noms de pays composés de plusieurs mots :
list(WorldCountries[WorldCountries["FRname"].apply(lambda x: " " in x)].FRname)
list(WorldCountries[WorldCountries["ENname"].apply(lambda x: " " in x)].ENname)
# remplacement par des noms plus brefs :
shorten_country_name_EN_list = {
    "Korea, Democratic People's Republic of": "North Korea",
    'Congo, The Democratic Republic of the':  "DR Congo",
    'Venezuela, Bolivarian Republic of':      "Venezuela",
    
    'Bolivia, Plurinational State of': "Bolivia",    
    'Tanzania, United Republic of':    "Tanzania",
 
    'Iran, Islamic Republic of': "Iran",
    'Moldova, Republic of':      "Moldova",
    'Syrian Arab Republic':      "Syria",

    'Palestine, State of':   "Palestine",
    'Russian Federation':    "Russia",
    'Korea, Republic of':    "South Korea",
    
    'Bonaire, Sint Eustatius and Saba': 'Bonaire',
    'Brunei Darussalam': 'Brunei',
    "Lao People's Democratic Republic": "Laos",
    'Micronesia, Federated States of': 'Micronesia',
    'Saint Helena, Ascension and Tristan da Cunha': "Saint Helena",
    'Saint Martin (French part)': 'Saint Martin (F)',
    'Sint Maarten (Dutch part)': 'Sint Maarten (D)',
}
shorten_country_name_FR_list = {
    'République du Congo': "Congo",
    'République démocratique du Congo': "RD Congo",
    'République dominicaine': "Dominique",
    'Timor-Leste': "Timor occidental",
    'Bonaire, Saint Eustache et Saba': 'Bonaire,',
    'Brunei Darussalam': 'Brunei',
    'Saint-Martin (partie française)': 'Saint-Martin (F)',
    'Saint-Martin (partie néerlandaise)': 'Saint-Martin (N)',
}
# création des nouvelles colonnes :
WorldCountries["ENusename"] = WorldCountries["ENname"].apply(
lambda x : shorten_country_name_EN_list[x] if (x in shorten_country_name_EN_list.keys()) else x)
WorldCountries["FRusename"] = WorldCountries["FRname"].apply(
lambda x : shorten_country_name_FR_list[x] if (x in shorten_country_name_FR_list.keys()) else x)

# vérifications :
WorldCountries[WorldCountries["ENusename"] != WorldCountries["ENname"]]
# WorldCountries[WorldCountries["FRusename"] != WorldCountries["FRname"]]

,alpha2,ENname,FRname,ENcontinent,FRcontinent,continentCode,svgFlag,ENusename,FRusename
alpha3,,,,,,,,,
BOL,BO,"Bolivia, Plurinational State of",Bolivie,South America,Amérique du Sud,SAM,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl...",Bolivia,Bolivie
BES,BQ,"Bonaire, Sint Eustatius and Saba","Bonaire, Saint Eustache et Saba",North America,Amérique du Nord,NAM,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl...",Bonaire,"Bonaire,"
BRN,BN,Brunei Darussalam,Brunei Darussalam,Asia,Asie,ASI,"<svg xmlns=""http://www.w3.org/2000/svg"" xmlns:...",Brunei,Brunei
COD,CD,"Congo, The Democratic Republic of the",République démocratique du Congo,Africa,Afrique,AFR,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl...",DR Congo,RD Congo
IRN,IR,"Iran, Islamic Republic of",Iran,Asia,Asie,ASI,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl...",Iran,Iran
LAO,LA,Lao People's Democratic Republic,Laos,Asia,Asie,ASI,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl...",Laos,Laos
FSM,FM,"Micronesia, Federated States of",Micronésie,Oceania,Océanie,OCE,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl...",Micronesia,Micronésie
MDA,MD,"Moldova, Republic of",Moldavie,Europe,Europe,EUR,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl...",Moldova,Moldavie
PRK,KP,"Korea, Democratic People's Republic of",Corée du Nord,Asia,Asie,ASI,"<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl...",North Korea,Corée du Nord


#####  Ajout des codes couleurs des continents, remise en ordre des colonnes et visualisation :

In [6]:
# coloration des continents :
# continentColor = {
#     "ASI": "rgb(241, 148, 138)", 
#     "EUR": "rgb(247, 220, 111)",
#     "AFR": "rgb(130, 224, 170)", 
#     "OCE": "rgb(195, 155, 211)", 
#     "NAM": "rgb(133, 193, 233)", 
#     "SAM": "rgb(240, 178, 122)",
#     "ANT": "rgb(255,255,255)",
#     "WOR": "rgb(200,200,200)",
# }
continentColor = {
    "ASI": "rgb( 239, 154, 154 )", 
    "EUR": "rgb( 255, 183, 77)", 
    "AFR": "rgb(  128, 203, 196  )", 
    "OCE": "rgb( 206, 147, 216 )", 
    "NAM": "rgb( 144, 202, 249 )", 
    "SAM": "rgb( 197, 225, 165 )",
    "ANT": "rgb(255,255,255)",
    "WOR": "rgb(200,200,200)",
}
# continentColor = {
#     "ASI": "rgb(255,128,128)", 
#     "EUR": "rgb(212,195,46)", 
#     "AFR": "rgb(155,255,128)", 
#     "OCE": "rgb(128,174,255)", 
#     "NAM": "rgb(132,206,209)", 
#     "SAM": "rgb(255,128,210)",
#     "ANT": "rgb(255,255,255)",
#     "WOR": "rgb(200,200,200)",
# }
WorldCountries["continentColor"] = WorldCountries["continentCode"].apply(
    lambda x: continentColor[x] if x in continentColor.keys() else "#fff")
WorldCountries = WorldCountries[["alpha2", "ENname", "FRname", "ENusename", "FRusename", "ENcontinent", "FRcontinent", "continentCode", "continentColor", "svgFlag"]]
WorldCountries.head()

,alpha2,ENname,FRname,ENusename,FRusename,ENcontinent,FRcontinent,continentCode,continentColor,svgFlag
alpha3,,,,,,,,,,
AFG,AF,Afghanistan,Afghanistan,Afghanistan,Afghanistan,Asia,Asie,ASI,"rgb( 239, 154, 154 )","<svg xmlns=""http://www.w3.org/2000/svg"" xmlns:..."
ALA,AX,Åland Islands,Îles Åland,Åland Islands,Îles Åland,Europe,Europe,EUR,"rgb( 255, 183, 77)","<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl..."
ALB,AL,Albania,Albanie,Albania,Albanie,Europe,Europe,EUR,"rgb( 255, 183, 77)","<svg xmlns=""http://www.w3.org/2000/svg"" xmlns:..."
DZA,DZ,Algeria,Algérie,Algeria,Algérie,Africa,Afrique,AFR,"rgb( 128, 203, 196 )","<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl..."
ASM,AS,American Samoa,Samoa américaines,American Samoa,Samoa américaines,Oceania,Océanie,OCE,"rgb( 206, 147, 216 )","<svg xmlns=""http://www.w3.org/2000/svg"" id=""fl..."


In [7]:
# pays sans drapeau exclu :
WorldCountriesExportCSV = WorldCountries[WorldCountries["svgFlag"] != "no_flag"]
print("Dimensions du tableau : ", WorldCountriesExportCSV.shape)
WorldCountries[WorldCountries["svgFlag"] == "no_flag"]

Dimensions du tableau :  (232, 10)


,alpha2,ENname,FRname,ENusename,FRusename,ENcontinent,FRcontinent,continentCode,continentColor,svgFlag
alpha3,,,,,,,,,,


## ligne à supprimer:

In [8]:
WorldCountriesExportCSV.to_csv(produitPath+"world_countries.csv") # 231 lignes

## Construction de la liste des pays exhaustive :

In [9]:
# pays de WorldCountries :
print("PAYS AVEC CODE ALPHA3")
print("Nombre de lignes dans WorldCountries: ", len(WorldCountries["alpha2"]))
print(", ".join(list(WorldCountries["alpha2"])))

print("=================================================")

# pays présents dans le fichier sourceSVGFlags :
flagList = list()
for file in os.listdir(sourcePath+sourceSVGFlags):
    filename = os.path.join(directory, file)
    code = filename.split("/")[-1][:-4].upper()
#     if code not in WorldCountries["alpha2"]:      
#         extraCodeList.append(code)
    flagList.append(code)
print("PAYS AVEC DRAPEAU ET CODE ALPHA2")
print("Nombre de lignes dans sourceSVGFlags: ", len(flagList))
print(", ".join(flagList))

print("=================================================")

# pays non présents dans WorldCountries mais présents dans sourceSVGFlags (qui ont un drapeau ) :
noAlpha3List = [c for c in flagList if c not in list(WorldCountries["alpha2"])]
print("PAYS AVEC DRAPEAU MAIS SANS CODE ALPHA3 NI NOM CONNU")
print("Nombre de lignes dans sourcesSVGFlags, pas dans WorldCountries: ", len(noAlpha3List))
print(", ".join(list(noAlpha3List)))

print("=================================================")

# pays non présents dans sourceSVGFlags mais présents dans WorldCountries (qui n'ont pas drapeau ) :
print("PAYS AVEC CODE ALPHA3 MAIS SANS CODE ALPHA2 NI DRAPEAU CONNU")
noFlagList = [c for c in list(WorldCountries.index) if WorldCountries.at[c, "alpha2"] not in flagList]
print("Nombre de lignes dans sourcesSVGFlags, pas dans WorldCountries: ", len(noFlagList))
for c in noFlagList:
    print(c, "| alpha2= ", WorldCountries.at[c, "alpha2"], "| ENname= ", WorldCountries.at[c, "ENname"])

PAYS AVEC CODE ALPHA3
Nombre de lignes dans WorldCountries:  232
AF, AX, AL, DZ, AS, AD, AO, AI, AQ, AG, AR, AM, AW, AU, AT, AZ, BS, BH, BD, BB, BY, BE, BZ, BJ, BM, BT, BO, BQ, BA, BW, BR, VG, BN, BG, BF, BI, KH, CM, CA, CV, CF, TD, CL, CN, CX, CO, KM, CG, CK, CR, CI, HR, CU, CW, CY, CZ, CD, DK, DJ, DM, DO, TL, EC, EG, SV, GQ, ER, EE, SZ, ET, FK, FO, FJ, FI, FR, GF, PF, GA, GM, GE, DE, GH, GR, GL, GD, GP, GT, GG, GN, GW, GY, HT, HN, HK, HU, IS, IN, ID, IR, IQ, IE, IM, IL, IT, JM, JP, JE, JO, KZ, KE, KI, KW, KG, LA, LV, LB, LS, LR, LY, LI, LT, LU, MO, MG, MW, MY, MV, ML, MT, MH, MQ, MR, MU, YT, MX, FM, MD, MN, ME, MS, MA, MZ, MM, NA, NR, NP, NL, AN, NC, NZ, NI, NE, NG, NU, KP, MK, NO, OM, PK, PW, PS, PA, PG, PY, PE, PH, PL, PT, PR, QA, RE, RO, RU, RW, SH, KN, LC, MF, PM, VC, WS, ST, SA, SN, RS, SC, SL, SG, SX, SK, SI, SB, SO, ZA, KR, SS, ES, LK, SD, SR, SJ, SE, CH, SY, TW, TJ, TZ, TH, TG, TO, TT, TN, TR, TM, TC, TV, UG, UA, AE, GB, US, VI, UY, UZ, VU, VE, VN, WF, EH, YE, ZM, ZW
PAYS AVE

##### Complément de WorldCountries pour y ajouter les pays sans code alpha3, mais avec code alpha2 et drapeau :

In [10]:
noAlpha3ListAddendum = {
    'FR-BZH': ["FR-BZH", "Brittany", "Bretagne", "Brittany", "Bretagne", "EUR"],
    'GB-WLS': ["GB-WLS", "Wales", "Pays de Galles", "Wales", "Pays de Galles", "EUR"],
    'SM': ["SMR", "San Marino", "Saint-Marin", "San Marino", "Saint-Marin", "EUR"],
    'SH-TA': ["SH-TA", "Saint Helena, Ascension and Tristan da Cunha", "Sainte-Hélène, Ascension et Tristan da Cunha", "Saint Helena", "Sainte Hélène", "AFR"],
    'TF': ["FR-TF", "French Southern and Antarctic Lands", "Terres australes et antarctiques françaises", "TAAF","TAAF", "ANT"],
    'IC': ["ICx", "Canary Islands", "Îles Canaries", "Canary Islands", "Îles Canaries", "EUR"],
    'TK': ["TKL", "Tokelau", "Tokelau", "Tokelau", "Tokelau", "OCE"],
    'HM': ["HMD", "Heard and McDonald Islands", "Îles Heard-et-MacDonald", "Heard and McDonald Islands", "Îles Heard-et-MacDonald", "ANT"],
    'GB-SCT':["GB-SCT", "Scotland", "Ecosse", "Scotland", "Ecosse", "EUR"],
    'GU': ["GUM", "Guam", "Guam", "Guam", "Guam", "OCE"],
    'MC': ["MCO", "Monaco", "Monaco", "Monaco", "Monaco", "EUR"],
    'NF': ["NFK", "Norfolk Island", "Île Norfolk", "Norfolk Island", "Île Norfolk", "OCE"],
    'CC': ["CCK", "Territory of Cocos Islands", "Territoire des îles Cocos", "Cocos Islands", "Îles Cocos", "OCE"],
    'UN': ["UNO", "United Nations", "Nations Unies", "United Nations", "Nations Unies", "WOR"],
    'GI': ["GIB", "Gibraltar", "Gibraltar", "Gibraltar", "Gibraltar", "EUR"],
    'KY': ["CYM", "Cayman Islands", "Îles Caïmans", "Cayman Islands", "Îles Caïmans", "NAM"],
    'EU': ["EUR", "European Union", "Union Européenne", "EU", "UE", "EUR"],
    'MP': ["MNP", "Commonwealth of the Northern Mariana Islands", "Îles Mariannes du Nord", "Northern Mariana Islands", "Îles Mariannes du Nord", "OCE"],
# 'XX': ["", "null", "null", "null", "null", ""], # drapeau blanc (à supprimer)
    'SH-AC': ["SH-AC", "Ascension Island", "Île de l'Ascension", "Ascension Island", "Île de l'Ascension", "AFR"],
    'GS': ["GS", "South Georgia and the South Sandwich Islands", "Géorgie du Sud-et-les îles Sandwich du Sud", "South Georgia and the South Sandwich Islands", "Géorgie du Sud-et-les îles Sandwich du Sud", "SAM"],
    'EAC': ["EAC", "East African Community", "Communauté de l'Afrique de l'Est", "East African Community", "Communauté de l'Afrique de l'Est", "AFR"],
    'ES-CT': ["ES-CT", "Catalonia", "Catalogne", "Catalonia", "Catalogne", "EUR"],
# 'CP': ["", "", "", ""], # drapeau : France
    'GB-NIR':["GB-NIR", "Northern Ireland", " Irlande du Nord", "Northern Ireland", " Irlande du Nord", "EUR"],
    'XK': ["XKx", "Kosovo", "Kosovo", "Kosovo", "Kosovo", "EUR"],
    'UM': ["UMx", "United States Minor Outlying Islands", "Îles mineures éloignées des États-Unis", "United States Minor Islands", "Îles mineures des États-Unis", "NAM"],
    'PN': ["PCN", "Pitcairn Islands", "Îles Pitcairn", "Pitcairn Islands", "Îles Pitcairn", "OCE"],
    'CEFTA': ["CEFTA", "Central European Free Trade Agreement", "Accord de libre-échange centre-européen", "ALECE", "CEFTA", "EUR"],
# 'DG': ["", "", "", ""], # erreur (doublon avec IOT)
    'GB-ENG':["GB-ENG", "England", "Angleterre", "England", "Angleterre", "EUR"],
    'ARAB': ["ARAB", "League of Arab States ", "Ligue des États arabes", "Arab League ", "Ligue arabe", "ASI"],
    'ES-PV': ["ES-PV", "Basque Country", "Pays Basque", "Basque Country", "Pays Basque", "EUR"],
    'BV': ["BVT", "Bouvet Island", "Île Bouvet", "Bouvet Island", "Île Bouvet", "ANT"],
    'IO': ["IOT", "British Indian Ocean Territory", "Territoire britannique de l'océan Indien", "BIOT", "TBOI", "ASI"],
    'BL': ["FR-BL", "Saint-Barthélemy", "Saint-Barthélemy", "Saint-Barthélemy", "Saint-Barthélemy", "NAM"], # drapeau : France
    'VA': ["VAT", "Vatican", "Vatican", "Vatican", "Vatican", "EUR"],
    'ES-GA':["ES-GA", "Galicia", "Galice", "Galicia", "Galice", "EUR"],
# 'AC': ["", "", "", "", "", ""], # redondant avec SH-AC (à supprimer)
# 'TA': ["", "", "", "", "", ""], # redondant avec SH-TA (à supprimer)
    'SH-HL': ["SH-HL", "Saint Helena", "Sainte-Hélène", "Saint Helena", "Sainte-Hélène", "AFR"],
    'PC': ["SPC", "Pacific Community", "Communauté du Pacifique", "SPC", "CPS", "OCE"],
}
# suppression des 5 lignes : XX, CP, DG, AC, TA

In [11]:
WorldCountriesExtra = pd.DataFrame(
    {
        "alpha3": [c[0] for c in noAlpha3ListAddendum.values()],
        "alpha2": noAlpha3ListAddendum.keys(),
        "ENname": [c[1] for c in noAlpha3ListAddendum.values()],
        "FRname": [c[2] for c in noAlpha3ListAddendum.values()],
        "ENusename": [c[3] for c in noAlpha3ListAddendum.values()],
        "FRusename": [c[4] for c in noAlpha3ListAddendum.values()],
        "ENcontinent": [continentId[c[5]][0] for c in noAlpha3ListAddendum.values()],
        "FRcontinent": [continentId[c[5]][1] for c in noAlpha3ListAddendum.values()],
        "continentCode": [c[5] for c in noAlpha3ListAddendum.values()],
        "continentColor": [continentColor[c[5]] for c in noAlpha3ListAddendum.values()],
        "svgFlag": [flags[c] for c in noAlpha3ListAddendum.keys()]
    },
).set_index("alpha3")
WorldCountriesExtra.shape

(37, 10)

##### Regroupement de WorldCountries avec WorldCountriesExtra (+36 lignes) :

In [12]:
WorldCountriesTotal = pd.concat([WorldCountries, WorldCountriesExtra])
WorldCountriesTotal.sort_index(inplace=True)
WorldCountriesTotal.shape
WorldCountriesTotal
", ".join(list(WorldCountriesTotal.index))

'ABW, AFG, AGO, AIA, ALA, ALB, AND, ANT, ARAB, ARE, ARG, ARM, ASM, ATA, ATG, AUS, AUT, AZE, BDI, BEL, BEN, BES, BFA, BGD, BGR, BHR, BHS, BIH, BLR, BLZ, BMU, BOL, BRA, BRB, BRN, BTN, BVT, BWA, CAF, CAN, CCK, CEFTA, CHE, CHL, CHN, CIV, CMR, COD, COG, COK, COL, COM, CPV, CRI, CUB, CUW, CXR, CYM, CYP, CZE, DEU, DJI, DMA, DNK, DOM, DZA, EAC, ECU, EGY, ERI, ES-CT, ES-GA, ES-PV, ESH, ESP, EST, ETH, EUR, FIN, FJI, FLK, FR-BL, FR-BZH, FR-TF, FRA, FRO, FSM, GAB, GB-ENG, GB-NIR, GB-SCT, GB-WLS, GBR, GEO, GGY, GHA, GIB, GIN, GLP, GMB, GNB, GNQ, GRC, GRD, GRL, GS, GTM, GUF, GUM, GUY, HKG, HMD, HND, HRV, HTI, HUN, ICx, IDN, IMN, IND, IOT, IRL, IRN, IRQ, ISL, ISR, ITA, JAM, JEY, JOR, JPN, KAZ, KEN, KGZ, KHM, KIR, KNA, KOR, KWT, LAO, LBN, LBR, LBY, LCA, LIE, LKA, LSO, LTU, LUX, LVA, MAC, MAF, MAR, MCO, MDA, MDG, MDV, MEX, MHL, MKD, MLI, MLT, MMR, MNE, MNG, MNP, MOZ, MRT, MSR, MTQ, MUS, MWI, MYS, MYT, NAM, NCL, NER, NFK, NGA, NIC, NIU, NLD, NOR, NPL, NRU, NZL, OMN, PAK, PAN, PCN, PER, PHL, PLW, PNG, PO

# Production des résultats :

### DataFrame :

In [13]:
# dataframe disponible :
WorldCountriesTotal.loc[["ARAB"]]

,alpha2,ENname,FRname,ENusename,FRusename,ENcontinent,FRcontinent,continentCode,continentColor,svgFlag
alpha3,,,,,,,,,,
ARAB,ARAB,League of Arab States,Ligue des États arabes,Arab League,Ligue arabe,Asia,Asie,ASI,"rgb( 239, 154, 154 )","<svg xmlns=""http://www.w3.org/2000/svg"" xml:sp..."


In [14]:
WorldCountriesTotal.loc["ARAB", "FRname"]

'Ligue des États arabes'

### CSV :

In [15]:
WorldCountriesTotal.to_csv(f"{produitPath}/dataset.csv")

### JSON :

In [16]:
WorldCountriesTotal.to_json(f"{produitPath}/dataset.json")

### JS :

In [17]:
countryId = {c: [
    WorldCountriesTotal.at[c, "alpha2"],
    WorldCountriesTotal.at[c, "ENname"],
    WorldCountriesTotal.at[c, "FRname"],
    WorldCountriesTotal.at[c, "ENusename"],
    WorldCountriesTotal.at[c, "FRusename"],
    WorldCountriesTotal.at[c, "ENcontinent"],
    WorldCountriesTotal.at[c, "FRcontinent"],
    WorldCountriesTotal.at[c, "continentCode"],
    WorldCountriesTotal.at[c, "continentColor"],
    WorldCountriesTotal.at[c, "svgFlag"],
] for c in WorldCountriesTotal.index}

In [18]:
with open(f"{produitPath}/dataset.js", 'w') as f:
    f.write("\n".join([
        f'const countryId = {countryId}',  # poids des flags : 348 kB
        "",
        ]))

# Nom des images des drapeaux (svg)

### dossier drapeaux svg référencés sous la forme alpha3 : 
#### > copie du dossier des drapeaux de type 'fr.svg' sous la forme 'fra.svg'

In [19]:
import shutil
newPath = f"{sourcePath}flag-icons_2023_alpha3"
try:
    os.mkdir(newPath)
except FileExistsError:
    print("Info : fichier {newPath} existe déjà")

for alpha3 in WorldCountriesTotal.index:
    try:
        alpha2 = WorldCountriesTotal.loc[alpha3, 'alpha2']
        shutil.copy(f"{sourcePath}{sourceSVGFlags}/{alpha2.lower()}.svg", 
                    f"{sourcePath}flag-icons_2023_alpha3/{alpha3.lower()}.svg")
        print(alpha3, '/', alpha2, end="|")
    except KeyError:
        print(alpha3, "NOALPHA2", end="|")

Info : fichier {newPath} existe déjà
ABW / AW|AFG / AF|AGO / AO|AIA / AI|ALA / AX|ALB / AL|AND / AD|ANT / AN|ARAB / ARAB|ARE / AE|ARG / AR|ARM / AM|ASM / AS|ATA / AQ|ATG / AG|AUS / AU|AUT / AT|AZE / AZ|BDI / BI|BEL / BE|BEN / BJ|BES / BQ|BFA / BF|BGD / BD|BGR / BG|BHR / BH|BHS / BS|BIH / BA|BLR / BY|BLZ / BZ|BMU / BM|BOL / BO|BRA / BR|BRB / BB|BRN / BN|BTN / BT|BVT / BV|BWA / BW|CAF / CF|CAN / CA|CCK / CC|CEFTA / CEFTA|CHE / CH|CHL / CL|CHN / CN|CIV / CI|CMR / CM|COD / CD|COG / CG|COK / CK|COL / CO|COM / KM|CPV / CV|CRI / CR|CUB / CU|CUW / CW|CXR / CX|CYM / KY|CYP / CY|CZE / CZ|DEU / DE|DJI / DJ|DMA / DM|DNK / DK|DOM / DO|DZA / DZ|EAC / EAC|ECU / EC|EGY / EG|ERI / ER|ES-CT / ES-CT|ES-GA / ES-GA|ES-PV / ES-PV|ESH / EH|ESP / ES|EST / EE|ETH / ET|EUR / EU|FIN / FI|FJI / FJ|FLK / FK|FR-BL / BL|FR-BZH / FR-BZH|FR-TF / TF|FRA / FR|FRO / FO|FSM / FM|GAB / GA|GB-ENG / GB-ENG|GB-NIR / GB-NIR|GB-SCT / GB-SCT|GB-WLS / GB-WLS|GBR / GB|GEO / GE|GGY / GG|GHA / GH|GIB / GI|GIN / GN|GLP / GP|GMB / GM|